In [ ]:
# ==========================================
# CELL 1: ENVIRONMENT SETUP & DYNAMIC BOUNDS
# ==========================================

import xarray as xr
import geopandas as gpd
import os
import time

# --- Directory Configuration ---
# Using the current working directory ensures the code remains portable
base_dir = os.getcwd()
base_output_dir = base_dir

# --- Climate Model Parameters ---
target_model = "ACCESS-CM2"
variables_to_process = ["pr", "tasmax", "tasmin", "rsds", "sfcWind"]

# Dictionary mapping each scenario to its specific year range
scenarios_to_process = {
    "historical": (1981, 2014),
    "ssp245": (2015, 2050),
    "ssp585": (2015, 2050)
}

# --- Dynamic Nyabarongo Bounding Box ---
print("Calculating precise extraction boundaries from shapefile...")
shapefile_path = os.path.join(base_dir, "nyabarongo.shp")

# Failsafe to ensure the shapefile is present before proceeding
if not os.path.exists(shapefile_path):
    raise FileNotFoundError(f"Shapefile missing at {shapefile_path}. Please verify the file exists.")

catchment = gpd.read_file(shapefile_path)

# Ensure the shapefile uses the standard WGS84 coordinate reference system
if catchment.crs != "EPSG:4326":
    catchment = catchment.to_crs("EPSG:4326")

# Automatically calculate the exact bounding box
# total_bounds returns [lon_min, lat_min, lon_max, lat_max]
bounds = catchment.total_bounds 

# Add a 0.25-degree "halo" (exactly 1 CMIP6 pixel) to guarantee edge coverage
halo = 0.25 
longitude_bounds = (bounds[0] - halo, bounds[2] + halo)
latitude_bounds = (bounds[1] - halo, bounds[3] + halo)

print("Environment setup complete. Variables initialized.")
print(f"Calculated Lat Bounds: {latitude_bounds}")
print(f"Calculated Lon Bounds: {longitude_bounds}")

Calculating precise extraction boundaries from shapefile...


FileNotFoundError: Shapefile missing at c:\Users\Baikania Amonison\Desktop\Nyabarongo-Catchment-Irrigation-Needs\notebooks\nyabarongo.shp. Please verify the file exists.

In [2]:
# ==========================================
# CELL 2: OPENDAP URL GENERATOR
# ==========================================

def generate_opendap_url(model, scenario, variable, year):
    """
    Constructs the precise OPeNDAP URL for a specific CMIP6 dataset on NASA's servers.
    """
    base_url = "https://ds.nccs.nasa.gov/thredds/dodsC/AMES/NEX/GDDP-CMIP6"
    ensemble = "r1i1p1f1"
    
    # Construct the exact filename based on NEX-GDDP-CMIP6 naming conventions
    filename = f"{variable}_day_{model}_{scenario}_{ensemble}_gn_{year}_v2.0.nc"
    
    # Combine into the full URL
    full_url = f"{base_url}/{model}/{scenario}/{ensemble}/{variable}/{filename}"
    
    return full_url

print("URL Generator function loaded.")

URL Generator function loaded.


In [3]:
# ==========================================
# CELL 3: OPENDAP EXTRACTION & AGGREGATION ENGINE (V3)
# ==========================================

def extract_and_aggregate_catchment_data(model, scenario, variable, start_year, end_year, lat_bnds, lon_bnds, output_dir, max_retries=4):
    
    os.makedirs(output_dir, exist_ok=True)
    temp_dir = os.path.join(output_dir, "temp_nc")
    os.makedirs(temp_dir, exist_ok=True)
    
    output_file = os.path.join(output_dir, f"Nyabarongo_{variable}_{scenario}_aggregated.nc")
    
    if os.path.exists(output_file):
        print(f"\n--- {output_file} already exists. Skipping. ---")
        return

    temp_files = []
    print(f"\n--- Starting extraction for {variable} ({scenario}) from {start_year} to {end_year} ---")
    
    # Modern time coder for the newest version of xarray
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

    for year in range(start_year, end_year + 1):
        dods_url = generate_opendap_url(model, scenario, variable, year)
        temp_filename = os.path.join(temp_dir, f"{variable}_{year}.nc")
        
        if os.path.exists(temp_filename):
            print(f" -> Found existing temp file for {year}. Skipping download.")
            temp_files.append(temp_filename)
            continue
            
        success = False

        for attempt in range(max_retries):
            try:
                print(f" Fetching {year} (Attempt {attempt + 1}/{max_retries})...")
                ds = xr.open_dataset(dods_url, engine='netcdf4', decode_times=time_coder)
                
                vars_to_drop = [v for v in ['time_bnds', 'lat_bnds', 'lon_bnds'] if v in ds.variables]
                if vars_to_drop:
                    ds = ds.drop_vars(vars_to_drop)
                
                subset = ds.sel(lat=slice(lat_bnds[0], lat_bnds[1]), lon=slice(lon_bnds[0], lon_bnds[1]))
                subset.to_netcdf(temp_filename)
                
                ds.close()
                subset.close()
                
                temp_files.append(temp_filename)
                print(f" -> Successfully saved {year}")
                success = True
                
                time.sleep(3)
                break
                
            except Exception as e:
                print(f" -> Error on attempt {attempt + 1} for {year}: {type(e).__name__} - {e}")
                if attempt < max_retries - 1:
                    print(" Waiting 5 seconds before retrying...")
                    time.sleep(5)
                    
        if not success:
            print(f" -> WARNING: Failed to fetch {year}. Data for this year will be missing.")

    print(f"\nAggregating all successfully downloaded years for {variable} ({scenario})...")
    if temp_files:
        try:
            # CRITICAL FIX: Using decode_times=time_coder instead of the deprecated use_cftime=True
            aggregated_ds = xr.open_mfdataset(temp_files, combine='by_coords', engine='netcdf4', decode_times=time_coder)
            aggregated_ds.load() 
            aggregated_ds.to_netcdf(output_file)
            print(f"Success! Aggregated data saved locally to: {output_file}")
            aggregated_ds.close()
            
            for file in temp_files:
                try:
                    os.remove(file)
                except OSError:
                    pass
            try:
                os.rmdir(temp_dir)
            except OSError:
                pass 
                
        except Exception as e:
            print(f"Error during aggregation: {e}")
    else:
        print(f"No files were successfully downloaded for {variable}.")

print("Extraction Engine (V3 - Time Coder Patched) loaded.")

Extraction Engine (V3 - Time Coder Patched) loaded.


In [4]:
# ==========================================
# CELL 4: EXECUTION MASTER LOOP
# ==========================================

# Iterate through each scenario defined in Cell 1
for scenario, (start_year, end_year) in scenarios_to_process.items():
    print(f"\n==========================================")
    print(f"INITIATING SCENARIO: {scenario.upper()}")
    print(f"==========================================")
    
    # Inside each scenario, process all 5 required variables
    for var in variables_to_process:
        extract_and_aggregate_catchment_data(
            model=target_model,
            scenario=scenario,
            variable=var,
            start_year=start_year,
            end_year=end_year,
            lat_bnds=latitude_bounds,
            lon_bnds=longitude_bounds,
            output_dir=base_output_dir,
            max_retries=4
        )

print("\n!!! ALL SCENARIOS AND VARIABLES PROCESSED COMPLETELY !!!")


INITIATING SCENARIO: HISTORICAL

--- Starting extraction for pr (historical) from 1981 to 2014 ---
 Fetching 1981 (Attempt 1/4)...
 -> Successfully saved 1981
 Fetching 1982 (Attempt 1/4)...
 -> Successfully saved 1982
 Fetching 1983 (Attempt 1/4)...
 -> Successfully saved 1983
 Fetching 1984 (Attempt 1/4)...
 -> Successfully saved 1984
 Fetching 1985 (Attempt 1/4)...
 -> Successfully saved 1985
 Fetching 1986 (Attempt 1/4)...
 -> Successfully saved 1986
 Fetching 1987 (Attempt 1/4)...
 -> Successfully saved 1987
 Fetching 1988 (Attempt 1/4)...
 -> Successfully saved 1988
 Fetching 1989 (Attempt 1/4)...
 -> Successfully saved 1989
 Fetching 1990 (Attempt 1/4)...
 -> Successfully saved 1990
 Fetching 1991 (Attempt 1/4)...
 -> Successfully saved 1991
 Fetching 1992 (Attempt 1/4)...
 -> Successfully saved 1992
 Fetching 1993 (Attempt 1/4)...
 -> Successfully saved 1993
 Fetching 1994 (Attempt 1/4)...
 -> Successfully saved 1994
 Fetching 1995 (Attempt 1/4)...
 -> Successfully saved 1995


In [1]:
# ==========================================
# CELL 6: MASTER DATA VALIDATOR
# ==========================================

import os
import xarray as xr
import warnings

# Suppress xarray FutureWarnings for a clean report
warnings.filterwarnings("ignore", category=FutureWarning)

def run_master_verification():
    base_dir = r"C:\Users\Baikania Amonison\Desktop\final data extraction and analysis"
    
    variables = ["pr", "tasmax", "tasmin", "rsds", "sfcWind"]
    scenarios = {
        "historical": (1981, 2014),
        "ssp245": (2015, 2050),
        "ssp585": (2015, 2050)
    }
    
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    all_clear = True
    
    print("\n" + "#"*50)
    print(" INITIATING FINAL DATASET VERIFICATION")
    print("#"*50)
    
    for scenario, (start_yr, end_yr) in scenarios.items():
        print(f"\n==========================================")
        print(f" SCENARIO: {scenario.upper()} ({start_yr} - {end_yr})")
        print(f"==========================================")
        
        for var in variables:
            file_name = f"Nyabarongo_{var}_{scenario}_aggregated.nc"
            file_path = os.path.join(base_dir, file_name)
            
            # 1. Check if file exists
            if not os.path.exists(file_path):
                print(f" ❌ [MISSING] {var.ljust(8)}: File does not exist.")
                all_clear = False
                continue
                
            try:
                ds = xr.open_dataset(file_path, engine='netcdf4', decode_times=time_coder)
                
                # 2. Check for missing years
                actual_years = set(ds['time'].dt.year.values)
                expected_years = set(range(start_yr, end_yr + 1))
                missing_years = sorted(list(expected_years - actual_years))
                
                # 3. Check chronological order
                is_monotonic = ds.indexes['time'].is_monotonic_increasing
                
                ds.close() # Free memory
                
                # Report Generation
                errors = []
                if missing_years:
                    errors.append(f"Missing years {missing_years}")
                if not is_monotonic:
                    errors.append("Timeline is NOT chronological (needs sorting)")
                    
                if not errors:
                    print(f" [PERFECT] {var.ljust(8)}: All {len(actual_years)} years present & chronologically sorted.")
                else:
                    print(f" [WARNING] {var.ljust(8)}: " + " | ".join(errors))
                    all_clear = False
                    
            except Exception as e:
                print(f" [ERROR]   {var.ljust(8)}: File corrupted or unreadable ({type(e).__name__})")
                all_clear = False
                
    print("\n" + "#"*50)
    if all_clear:
        print(" SYSTEM AUDIT PASSED! ALL DATA IS PRISTINE AND READY FOR ANALYSIS.")
    else:
        print(" SYSTEM AUDIT FAILED. PLEASE REVIEW THE WARNINGS ABOVE.")
    print("#"*50 + "\n")

# Execute
run_master_verification()


##################################################
 INITIATING FINAL DATASET VERIFICATION
##################################################

 SCENARIO: HISTORICAL (1981 - 2014)
 [PERFECT] pr      : All 34 years present & chronologically sorted.
 [PERFECT] tasmax  : All 34 years present & chronologically sorted.
 [PERFECT] tasmin  : All 34 years present & chronologically sorted.
 [PERFECT] rsds    : All 34 years present & chronologically sorted.
 [PERFECT] sfcWind : All 34 years present & chronologically sorted.

 SCENARIO: SSP245 (2015 - 2050)
 [PERFECT] pr      : All 36 years present & chronologically sorted.
 [PERFECT] tasmax  : All 36 years present & chronologically sorted.
 [PERFECT] tasmin  : All 36 years present & chronologically sorted.
 [PERFECT] rsds    : All 36 years present & chronologically sorted.
 [PERFECT] sfcWind : All 36 years present & chronologically sorted.

 SCENARIO: SSP585 (2015 - 2050)
 [PERFECT] pr      : All 36 years present & chronologically sorted.
 [PERFE

In [2]:
# ==========================================
# CELL 5: STRICT CLIP & DYNAMIC DEM ELEVATION EXTRACTION
# ==========================================
import pandas as pd
import numpy as np
import rioxarray
import geopandas as gpd
from shapely.geometry import Point
import os

# 1. Define your exact base directory
base_dir = r"C:\Users\Baikania Amonison\Desktop\final data extraction and analysis"

# 2. Load the Catchment Shapefile and force standard GPS coordinates (WGS84)
catchment = gpd.read_file(os.path.join(base_dir, "nyabarongo.shp"))
catchment = catchment.to_crs(epsg=4326)

# Load the 30m SRTM DEM and extract its exact spatial language (CRS)
dem_path = os.path.join(base_dir, "srtm_30m_nyabarongo.tif") 
dem = rioxarray.open_rasterio(dem_path)
dem_crs = dem.rio.crs

# 3. Load the Raw Excel Data 
excel_path = os.path.join(base_dir, "All Rainfall stations_1981-2021_Data.xlsx")
df_raw = pd.read_excel(excel_path, header=None)

# 4. Extract Variables based on Excel structure
station_names = df_raw.iloc[0, 1:].values
longitudes = df_raw.iloc[1, 1:].values
latitudes = df_raw.iloc[2, 1:].values

# Initialize tracking lists
valid_stations_idx = []
kept_station_names = []
station_elevations = []
station_pressures = []

# 5. Loop through stations to clip and extract precise elevation
for i in range(len(station_names)):
    try:
        lon = float(longitudes[i])
        lat = float(latitudes[i])
    except ValueError:
        continue
    
    if pd.notna(lon) and pd.notna(lat):
        station_point = Point(lon, lat)
        
        # Keep ONLY stations that physically fall inside the Nyabarongo polygon
        if catchment.geometry.contains(station_point).any():
            valid_stations_idx.append(i + 1)
            kept_station_names.append(station_names[i])
            
            # --- THE FIX: CRS TRANSLATION ---
            # Create a geographic object for the station in WGS84
            pt_wgs84 = gpd.GeoSeries([station_point], crs="EPSG:4326")
            
            # Translate that point into the DEM's specific coordinate system
            pt_dem_crs = pt_wgs84.to_crs(dem_crs).iloc[0]
            
            # Extract exact elevation (z) using the translated coordinates
            try:
                z = float(dem.sel(x=pt_dem_crs.x, y=pt_dem_crs.y, method="nearest"))
                
                # Failsafe: Rwanda is high-altitude. If z <= 0, it hit a NoData pixel.
                if z <= 0:
                    z = np.nan
            except:
                z = np.nan
            
            # Calculate dynamic Atmospheric Pressure (P) in kPa
            if not np.isnan(z):
                p_val = 101.3 * (((293.0 - 0.0065 * z) / 293.0) ** 5.26)
            else:
                p_val = np.nan
                
            station_elevations.append(z)
            station_pressures.append(p_val)

print(f"Strict Clip Success: Isolated {len(valid_stations_idx)} stations.")

# 6. Filter and Save
if len(valid_stations_idx) > 0:
    columns_to_keep = [0] + valid_stations_idx
    df_clipped = df_raw.iloc[:, columns_to_keep].copy()

    metadata_df = pd.DataFrame({
        'Station_Name': kept_station_names,
        'Elevation_m': station_elevations,
        'Atm_Pressure_kPa': station_pressures
    })

    final_output_csv = os.path.join(base_dir, "Observed_Rainfall_Clipped.csv")
    metadata_output_csv = os.path.join(base_dir, "Station_Elevation_Metadata.csv")

    df_clipped.to_csv(final_output_csv, index=False, header=False)
    metadata_df.to_csv(metadata_output_csv, index=False)

    print(f"Clipped rainfall saved to: {final_output_csv}")
    print(f"Elevation & Pressure metadata saved to: {metadata_output_csv}")
else:
    print("WARNING: 0 stations clipped.")

Strict Clip Success: Isolated 64 stations.
Clipped rainfall saved to: C:\Users\Baikania Amonison\Desktop\final data extraction and analysis\Observed_Rainfall_Clipped.csv
Elevation & Pressure metadata saved to: C:\Users\Baikania Amonison\Desktop\final data extraction and analysis\Station_Elevation_Metadata.csv


In [5]:
# ==========================================
# CELL 5: STRICT CLIP & DYNAMIC DEM ELEVATION EXTRACTION
# ==========================================
import pandas as pd
import numpy as np
import rioxarray
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer
import os

# 1. Define your exact base directory
base_dir = r"C:\Users\Baikania Amonison\Desktop\final data extraction and analysis"

# 2. Load the Catchment Shapefile and force standard GPS coordinates (WGS84)
catchment = gpd.read_file(os.path.join(base_dir, "nyabarongo.shp"))
catchment = catchment.to_crs(epsg=4326)

# Load the 30m SRTM DEM
dem_path = os.path.join(base_dir, "srtm_30m_nyabarongo.tif") 
dem = rioxarray.open_rasterio(dem_path)
dem_crs = dem.rio.crs

# --- THE FIX: STRICT COORDINATE TRANSLATOR ---
# This translates your GPS degrees into the DEM's specific "TM Rwanda" meters
transformer = Transformer.from_crs("EPSG:4326", dem_crs, always_xy=True)

# 3. Load the Raw Excel Data 
excel_path = os.path.join(base_dir, "All Rainfall stations_1981-2021_Data.xlsx")
df_raw = pd.read_excel(excel_path, header=None)

# 4. Extract Variables based on Excel structure
station_names = df_raw.iloc[0, 1:].values
longitudes = df_raw.iloc[1, 1:].values
latitudes = df_raw.iloc[2, 1:].values

# Initialize tracking lists
valid_stations_idx = []
kept_station_names = []
station_elevations = []
station_pressures = []

# 5. Loop through stations to clip and extract precise elevation
for i in range(len(station_names)):
    try:
        lon = float(longitudes[i])
        lat = float(latitudes[i])
    except ValueError:
        continue
    
    if pd.notna(lon) and pd.notna(lat):
        station_point = Point(lon, lat)
        
        # Keep ONLY stations that physically fall inside the Nyabarongo polygon
        if catchment.geometry.contains(station_point).any():
            valid_stations_idx.append(i + 1)
            kept_station_names.append(station_names[i])
            
            # Translate the GPS coordinate into the DEM's meter coordinates
            dem_x, dem_y = transformer.transform(lon, lat)
            
            # Extract exact elevation (z)
            try:
                # Use the translated X and Y
                z = float(dem.sel(x=dem_x, y=dem_y, method="nearest").values[0])
                
                # Failsafe: Your diagnostic showed NoData is -9996. Filter it out.
                if z < 0:
                    z = np.nan
            except:
                z = np.nan
            
            # Calculate dynamic Atmospheric Pressure (P) in kPa
            if not np.isnan(z):
                p_val = 101.3 * (((293.0 - 0.0065 * z) / 293.0) ** 5.26)
            else:
                p_val = np.nan
                
            station_elevations.append(z)
            station_pressures.append(p_val)

print(f"Strict Clip Success: Isolated {len(valid_stations_idx)} stations.")

# 6. Filter and Save
if len(valid_stations_idx) > 0:
    columns_to_keep = [0] + valid_stations_idx
    df_clipped = df_raw.iloc[:, columns_to_keep].copy()

    metadata_df = pd.DataFrame({
        'Station_Name': kept_station_names,
        'Elevation_m': [round(e, 2) if pd.notna(e) else np.nan for e in station_elevations],
        'Atm_Pressure_kPa': [round(p, 3) if pd.notna(p) else np.nan for p in station_pressures]
    })

    final_output_csv = os.path.join(base_dir, "Observed_Rainfall_Clipped.csv")
    metadata_output_csv = os.path.join(base_dir, "Station_Elevation_Metadata.csv")

    df_clipped.to_csv(final_output_csv, index=False, header=False)
    metadata_df.to_csv(metadata_output_csv, index=False)

    print(f"Clipped rainfall saved to: {final_output_csv}")
    print(f"Elevation & Pressure metadata saved to: {metadata_output_csv}")

Strict Clip Success: Isolated 64 stations.
Clipped rainfall saved to: C:\Users\Baikania Amonison\Desktop\final data extraction and analysis\Observed_Rainfall_Clipped.csv
Elevation & Pressure metadata saved to: C:\Users\Baikania Amonison\Desktop\final data extraction and analysis\Station_Elevation_Metadata.csv
